# Round trip: every assembled data source for one tile / year

Reconstructs a raster for **every column** of the assembled 1 km panel for a
single `(ix, iy)` tile and a single `year`, straight from `pixel_id`, and lays
them all out in one figure. This is the round-trip check that the flat panel can
be folded back to the pixel grid it came from.

The panel is written by the DuckDB SQL assembly engine
(`src/data/assemble/sql_engine.py`) onto the canonical EPSG:6933 EASE grid
(`src/data/common/geobox/canonical.py`), tiled 2048x2048
(`src/data/assemble/tiles.py`). `pixel_id` packs `[ix:16 | iy:16 | local:32]`;
at the native `grid_label="1km"` resolution used here, `local` is a plain
row-major index into the tile, which is what this notebook decodes.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import pyproj
from odc.geo.xr import ODCExtensionDa  # noqa: F401  (registers the .odc accessor)

from src.data.common.geobox.canonical import canonical_ease_geobox
from src.data.assemble.tiles import create_tile_geobox
from src.data.assemble.utils import make_pixel_ids, geobox_spatial_dims
from src.data.assemble.constants import GRID_RESOLUTIONS_M, DEFAULT_TILE_SIZE

In [ ]:
# --- parameters --------------------------------------------------------------
PROJECT = "/scicore/home/meiera/schulz0022/projects/growth-and-temperature"
DATA = f"{PROJECT}/data_nobackup"

ix, iy = 0, 0          # tile index into the canonical EASE tile grid
year = 2013            # panel year to plot (ignored for time-invariant columns)
grid_label = "1km"     # keep at "1km" -- coarser grids pack a ragged local
                        # pixel index this notebook does not decode
shake_label = "base"

# COPY ... PARTITION_BY (ix, iy) FILENAME_PATTERN 'data_{i}' -> data_0.parquet,
# data_1.parquet, ... under multiple writer threads (src/data/assemble/sql_engine.py)
assembled_tile = (
    f"{DATA}/assembled/grid={grid_label}/shake={shake_label}"
    f"/ix={ix}/iy={iy}/data_*.parquet"
)

In [ ]:
# --- load the panel tile + the geobox it was cut from ----------------------
parquet_tile = pd.read_parquet(assembled_tile)

target_geobox = canonical_ease_geobox(resolution_m=GRID_RESOLUTIONS_M[grid_label])
tile = create_tile_geobox(target_geobox, DEFAULT_TILE_SIZE, ix, iy)
dim_y, dim_x = geobox_spatial_dims(tile)  # ('y', 'x') -- EASE6933 is projected, not lat/lon

print("panel rows :", len(parquet_tile))
print("columns    :", list(parquet_tile.columns))
if "year" in parquet_tile.columns:
    yrs = np.sort(parquet_tile["year"].dropna().unique())
    print("years      :", yrs.min(), "..", yrs.max(), f"({len(yrs)} unique)")

In [ ]:
# --- pixel_id -> (y, x) for this tile ---------------------------------------
pixel_id_ds = make_pixel_ids(ix, iy, tile)
conversion_df = (
    pixel_id_ds["pixel_id"].to_dataframe().reset_index()[["pixel_id", dim_y, dim_x]]
)

# tile footprint in lon/lat, just for the figure title
transformer = pyproj.Transformer.from_crs(tile.crs, "EPSG:4326", always_xy=True)
(lon0, lon1), (lat0, lat1) = transformer.transform(
    [tile.boundingbox.left, tile.boundingbox.right],
    [tile.boundingbox.bottom, tile.boundingbox.top],
)

In [ ]:
# --- pick the columns to plot -------------------------------------------------
id_cols = {"pixel_id", "year", "ix", "iy", dim_y, dim_x}
value_cols = [c for c in parquet_tile.columns if c not in id_cols]

panel = parquet_tile
if "year" in panel.columns:
    panel = panel[panel["year"] == year]
    if panel.empty:
        raise ValueError(f"no panel rows for year={year}")

# drop columns that are entirely missing for this tile/year
value_cols = [c for c in value_cols if panel[c].notna().any()]

# object / category columns -> integer codes so they still render
encoded = panel[["pixel_id"]].copy()
cat_cols = []
for c in value_cols:
    s = panel[c]
    if s.dtype == object or str(s.dtype).startswith("category"):
        encoded[c] = pd.factorize(s)[0].astype("float32")
        encoded.loc[s.isna().values, c] = np.nan
        cat_cols.append(c)
    else:
        encoded[c] = s.astype("float32")

print(f"{len(value_cols)} columns to plot; categorical (code-encoded): {cat_cols}")

In [ ]:
# --- fold every column back onto the tile grid ------------------------------
grid = (
    conversion_df.merge(encoded, on="pixel_id", how="left")
    .set_index([dim_y, dim_x])[value_cols]
    .to_xarray()
)
grid = grid.odc.assign_crs(tile.crs)
grid

In [ ]:
# --- one imshow per source -------------------------------------------------
ncols = 3
nrows = int(np.ceil(len(value_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.4 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, col in zip(axes, value_cols):
    da = grid[col]
    cmap = "tab20" if col in cat_cols else "viridis"
    da.plot.imshow(ax=ax, robust=True, cmap=cmap, add_labels=False)
    ax.set_title(col + (" (codes)" if col in cat_cols else ""), fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

for ax in axes[len(value_cols):]:
    ax.set_visible(False)

fig.suptitle(
    f"assembled {grid_label} panel  -  tile ix={ix} iy={iy}  -  year {year}\n"
    f"lon [{lon0:.2f}, {lon1:.2f}]  lat [{lat0:.2f}, {lat1:.2f}]",
    y=1.02,
)
fig.tight_layout()